# 04 - Explainability

SHAP attribution, interaction analysis and threshold discovery for the two
models the API serves (`aquanexus.ml.explainer`, ML_STRATEGY.md §7.2).

The models explained here are the **saved artefacts** in `data/models/`, not
fresh fits - so what follows describes what the API actually returns.

Two things this notebook is at pains to separate:

- what the model does (SHAP answers this, and only this);
- what the river does (SHAP does **not** answer this, and on a collinear
  feature set it can be actively misleading about it).

In [ ]:
import sys; sys.path.insert(0, '../src')
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from aquanexus.config import settings
from aquanexus.data.dataset import (DO_FEATURES, build_state_vectors,
                                    build_water_quality_dataset, feature_columns)
from aquanexus.data.loader import filter_stations, load_many
from aquanexus.ml.explainer import HabitatExplainer
from aquanexus.ml.models import HabitatPredictor

pd.set_option('display.width', 200)

In [ ]:
files = sorted(glob.glob(str(settings.RAW_DIR / 'waterquality' / 'saitama_*.xlsx')))
observations = filter_stations(load_many(files), water_body=settings.RIVER_NAME_JA)
sweep = pd.read_csv(settings.PROCESSED_DIR / 'ayase_flow_sweep.csv')

water = build_water_quality_dataset(observations, sweep)
do_features = [f for f in DO_FEATURES if f in water.columns]

do_model = HabitatPredictor.load(settings.MODELS_DIR / 'dissolved_oxygen_v1.joblib')
print(f'{do_model.config.model_type} on {len(do_model.feature_names)} features, '
      f'output clipped to {do_model.config.output_range} mg/L')

## 1. Which explainer, and why it is not `TreeExplainer`

ML_STRATEGY §7.2 Step 2 specifies `TreeExplainer` - it assumed XGBoost would be
the chosen model. On the observed-DO target Ridge cross-validates better
(`03_model_training.ipynb`), and `TreeExplainer` cannot explain a Ridge
pipeline. `HabitatExplainer` dispatches on model type instead: Tree for
ensembles, Kernel for the imputer-scaler-Ridge pipeline.

The spec also asks for 8,760 background records. There are 138 rows in total,
so the background is the training data itself, k-means summarised.

In [ ]:
explainer = HabitatExplainer(do_model).fit_explainer(water[do_features],
                                                     max_background=50)
shap_values = explainer.compute(water[do_features])
print(f'SHAP values {shap_values.shape}, expected value '
      f'{explainer.expected_value:.2f} mg/L '
      f'(mean observed {water.dissolved_oxygen.mean():.2f} mg/L)')

## 2. Feature importance - with the caveat that comes before the table

Mean |SHAP| ranks how hard a feature moves predictions; the signed mean says
which way. Both are computed on the shipped model over all 138 observations.

In [ ]:
importance = explainer.feature_importance()
importance.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(6.6, 3.4))
colours = ['#c0392b' if v < 0 else '#2e86ab' for v in importance.mean_shap[::-1]]
ax.barh(importance.feature[::-1], importance.mean_abs_shap[::-1], color=colours)
ax.set_xlabel('mean |SHAP| (mg/L)')
ax.set_title('Dissolved oxygen: attribution on the shipped Ridge model',
             loc='left', fontweight='bold')
ax.text(0.98, 0.05, 'blue = raises DO on average\nred = lowers DO on average',
        transform=ax.transAxes, ha='right', fontsize=8, color='#5a6472')
plt.tight_layout()

## 3. Why those ranks cannot be read as a league table

Every hydraulic feature is derived from discharge through the same HEC-RAS
model, so depth, velocity, top width and Froude number are near-duplicates of
one another. Water temperature and DO saturation are related by a physical
equation, at r = -0.99.

SHAP still sums correctly to each prediction. But **how it divides that total
between collinear features is arbitrary** - swapping which of a pair the model
leans on changes the ranking without changing a single prediction.

In [ ]:
pairs = explainer.collinearity(threshold=0.9)
print(f'{len(pairs)} feature pairs correlate at |r| >= 0.9 across all 138 observations')
pairs.round(3)

The clearest symptom is in section 5 below: `water_temp` shows almost no
marginal response while `do_saturation` - a deterministic function of water
temperature - shows the largest of any feature. The model routes the thermal
signal through one of the pair. Reading either in isolation understates it.

**The hydraulic features should be read as one combined contribution**, not
ranked against each other.

## 4. Interactions (§7.2 Step 6) - and what the data cannot answer

The method splits the explained rows into high/low bins on two features and
compares the both-adverse corner against what the two one-at-a-time changes
predict additively. A negative interaction means the combination is worse than
the sum of its parts.

In [ ]:
candidates = [('water_temp', 'discharge'),
              ('water_temp', 'do_saturation'),
              ('discharge', 'reach_velocity'),
              ('water_temp', 'reach_depth')]

rows = []
for a, b in candidates:
    result = explainer.interaction_analysis(a, b)
    rows.append({'pair': f'{a} x {b}', 'correlation': result['correlation'],
                 'identifiable': result['identifiable'],
                 'interaction': result['interaction'],
                 'note': result.get('reason', result.get('warning', ''))})

pd.DataFrame(rows).round(3)

**Two pairs cannot be answered from this data at all.** When two features
correlate at 0.9+, one corner of the 2x2 design is empty - "high discharge,
low velocity" never happens - so the interaction is unidentifiable. The
explainer returns `identifiable: False` with the reason rather than a bare NaN,
because a NaN reads as "no interaction" when the truth is "cannot tell".

### Correction to the Phase 2b result

The Phase 2b run reported water temperature x discharge as **synergistic at
-1.02 mg/L** - warm water plus high flow depressing oxygen more than the parts
added - and `README.md`, `docs/ML_METHODOLOGY.md` and `DEVLOG.md` carried it as
a finding. It came from explaining a 48-row subset, which is one station
(52内匠橋). Over all 138 observations the same call returns a **positive**
interaction. Per station:

In [ ]:
stability = []
for station, block in water.groupby('station'):
    explainer.compute(block[do_features])
    result = explainer.interaction_analysis('water_temp', 'discharge')
    stability.append({'subset': station, 'n': result['n'],
                      'correlation': result['correlation'],
                      'interaction': result['interaction']})

explainer.compute(water[do_features])  # restore the full explained set
pooled = explainer.interaction_analysis('water_temp', 'discharge')
stability.append({'subset': 'all stations', 'n': pooled['n'],
                  'correlation': pooled['correlation'],
                  'interaction': pooled['interaction']})
pd.DataFrame(stability).round(3)

The sign flips between stations, from -0.93 to +1.72 mg/L, and the pooled
estimate is small and positive. **The synergy is not established.** With 18-48
observations per station and a temperature-discharge correlation that is itself
unstable (+0.07 to +0.69), this design cannot resolve an interaction of that
size; the -1.02 figure describes one station on one subset, not the river.

The ecological story behind it stays plausible - storm load arriving with warm
water is a real mechanism in urban lowland rivers - but plausible is not
measured, and it should not have been reported as a finding. Reporting it as
one is the mistake this cell exists to undo.

In [ ]:
cells = np.array([[pooled['low_low'], pooled['low_high']],
                  [pooled['high_low'], pooled['high_high']]])

fig, ax = plt.subplots(figsize=(5.0, 3.6))
image = ax.imshow(cells, cmap='RdYlBu', vmin=cells.min(), vmax=cells.max())
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{cells[i, j]:.2f}', ha='center', va='center', fontsize=11)
ax.set_xticks([0, 1], ['low discharge', 'high discharge'])
ax.set_yticks([0, 1], ['cool water', 'warm water'])
ax.set_title(f'mean predicted DO (mg/L), all 138 observations\n'
             f'interaction {pooled["interaction"]:+.2f} mg/L - not established',
             loc='left', fontweight='bold', fontsize=10)
fig.colorbar(image, ax=ax, shrink=0.8, label='mg/L')
plt.tight_layout()

## 5. Threshold discovery (§7.2 Step 5)

Each feature is swept across its observed 1st-99th percentile range with every
other feature held at its median. `response_span` is how far the prediction
moves across that sweep; a feature that barely moves it has no threshold worth
quoting, however sharp its steepest point looks.

In [ ]:
thresholds = pd.DataFrame([
    {k: v for k, v in explainer.threshold_discovery(water, feature).items()
     if k not in ('grid', 'prediction', 'gradient')}
    for feature in do_features
]).sort_values('response_span', ascending=False).reset_index(drop=True)
thresholds.round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
for ax, feature in zip(axes, ['do_saturation', 'discharge', 'water_temp'],
                       strict=True):
    sweep_result = explainer.threshold_discovery(water, feature)
    ax.plot(sweep_result['grid'], sweep_result['prediction'], color='#2e86ab', lw=1.8)
    ax.axvline(sweep_result['threshold'], color='#c0392b', ls='--', lw=1.1,
               label=f"steepest at {sweep_result['threshold']:.2f}")
    ax.set_xlabel(feature)
    ax.set_ylabel('predicted DO (mg/L)')
    ax.set_title(f"span {sweep_result['response_span']:.2f} mg/L"
                 f"{'' if sweep_result['is_influential'] else '  (not influential)'}",
                 loc='left', fontsize=9, fontweight='bold')
    ax.legend(frameon=False, fontsize=8)
plt.tight_layout()

These are partial-dependence curves. They describe **the model**, not the
river: holding correlated features at their median produces combinations that
may never physically occur - a high discharge at median velocity, for instance,
is not a state this reach can be in.

`water_temp` is flagged not influential here purely because of the collinearity
in section 3, not because temperature does not matter. It matters most of all;
the model reads it through `do_saturation`.

## 6. One prediction, explained

The lowest dissolved oxygen in the modelling set - warm September water at the
downstream station.

In [ ]:
worst = water.loc[[water.dissolved_oxygen.idxmin()]]
print(f"{worst.station.iloc[0]}  {worst.timestamp.iloc[0]:%Y-%m-%d}  "
      f"{worst.water_temp.iloc[0]:.1f} °C  "
      f"discharge {worst.discharge.iloc[0]:.1f} m3/s  "
      f"observed DO {worst.dissolved_oxygen.iloc[0]:.1f} mg/L "
      f"({100 * worst.dissolved_oxygen.iloc[0] / worst.do_saturation.iloc[0]:.0f}% "
      f"of saturation)")
print(explainer.explain_prediction(worst[do_features], top=6))

The attribution is readable - saturation at 27.6 °C pulls the prediction down
hardest - but note the gap between the prediction and the measured 3.0 mg/L.
**The model does not reach the extremes**, and the extremes are the states a
habitat diagnosis exists to catch. That failure is quantified by flow band in
`05_model_validation.ipynb`.

## 7. The synthetic model, for contrast

The same machinery on the HSI model - which is a tree ensemble, so it takes the
`TreeExplainer` path and runs on all 7,314 rows.

In [ ]:
state = build_state_vectors(observations, sweep)
features = feature_columns(state)
hsi_model = HabitatPredictor.load(settings.MODELS_DIR / 'hsi_v1.joblib')

hsi_explainer = HabitatExplainer(hsi_model).fit_explainer(state[features])
hsi_explainer.compute(state[features].sample(1500, random_state=42))
hsi_importance = hsi_explainer.feature_importance().head(8)
hsi_importance.round(4)

Hydraulics dominate, and dissolved oxygen barely registers. That is **not** an
ecological finding - it is a property of how the dataset was built. Chemistry
is held constant along the reach on a given date while 53 cross-sections span
depths of 0.1-5.6 m, so geometry has far more variance to explain than
chemistry does.

Explaining a model trained on generated labels tells you what the label
function does. It is a useful check that the pipeline is wired correctly, and
it is nothing more than that.

## Carried forward

- **The -1.02 mg/L temperature x discharge synergy does not hold.** It was a
  single-station result; pooled over all 138 observations the sign reverses and
  the effect is small. `README.md`, `docs/ML_METHODOLOGY.md` and `DEVLOG.md`
  are corrected to match this notebook. Nothing else in the project depended on
  it - the API does not serve interactions - but it was a headline claim.
- **Ten collinear pairs** mean individual SHAP ranks on the DO model are not
  trustworthy in isolation. Any future feature-selection work has to deal with
  this first. (The Phase 2b run reported twelve; that count was over one
  station's 48 rows, and it moves with the subset explained.)
- **2 of 4 interactions are unidentifiable** from this data. Answering them
  needs observations that break the discharge-velocity coupling - a second
  reach, or a gauged event.
- The saved SHAP artefact (`data/models/shap_values_do.npy`) covers that same
  48-row subset; this notebook recomputes over all 138.
- Debt unchanged from `03_model_training.ipynb`: 4 constrictions still in the
  geometry (RS 12500 / 14000 / 18000 / 24000), Manning's n uncalibrated, and
  the low-flow failure quantified next door in `05_model_validation.ipynb`.